In [2]:
# Cell 1: Imports, Setup and API Connection Test (Groq - Free)

import os
import pandas as pd
import numpy as np
from groq import Groq
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load API key from .env file
load_dotenv()
api_key = os.getenv('GROQ_API_KEY')

# Verify key loaded
if api_key:
    print(f"API Key loaded: {api_key[:15]}...{api_key[-4:]}")
else:
    print("ERROR: API Key not found - check your .env file!")

# Initialize Groq client
client = Groq(api_key=api_key)

# Quick connection test
print("\nTesting API connection...")
test_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "Say exactly this: F1 Race Engineer AI is online and ready!"
        }
    ]
)

print(f"API Response: {test_response.choices[0].message.content}")
print(f"Model used: {test_response.model}")
print(f"Input tokens: {test_response.usage.prompt_tokens}")
print(f"Output tokens: {test_response.usage.completion_tokens}")
print("\nAll systems go - Ready to build the AI Race Engineer!")

API Key loaded: gsk_wnyLBEgziTw...OQhi

Testing API connection...
API Response: F1 Race Engineer AI is online and ready!
Model used: llama-3.3-70b-versatile
Input tokens: 49
Output tokens: 11

All systems go - Ready to build the AI Race Engineer!


In [3]:
# Cell 2: Load F1 Data and Build Race Context

print("Loading F1 data for AI Race Engineer...")
print("="*60)

# Load all datasets
races        = pd.read_csv('../data/races.csv')
results      = pd.read_csv('../data/results.csv')
drivers      = pd.read_csv('../data/drivers.csv')
constructors = pd.read_csv('../data/constructors.csv')
pit_stops    = pd.read_csv('../data/pit_stops.csv')
qualifying   = pd.read_csv('../data/qualifying.csv')
circuits     = pd.read_csv('../data/circuits.csv')
driver_standings     = pd.read_csv('../data/driver_standings.csv')
constructor_standings = pd.read_csv('../data/constructor_standings.csv')

# Filter to modern era
modern_races   = races[races['year'].between(2020, 2024)].copy()
modern_results = results[results['raceId'].isin(modern_races['raceId'])].copy()
modern_pitstops = pit_stops[pit_stops['raceId'].isin(modern_races['raceId'])].copy()
modern_quali   = qualifying[qualifying['raceId'].isin(modern_races['raceId'])].copy()

print(f"Races loaded       : {modern_races.shape[0]}")
print(f"Results loaded     : {modern_results.shape[0]}")
print(f"Pit stops loaded   : {modern_pitstops.shape[0]}")
print(f"Qualifying loaded  : {modern_quali.shape[0]}")

# ─────────────────────────────────────────────────────────────
# 1. DRIVER STATS SUMMARY
# ─────────────────────────────────────────────────────────────

driver_info = drivers[['driverId', 'forename', 'surname', 'nationality']].copy()
driver_info['full_name'] = driver_info['forename'] + ' ' + driver_info['surname']

merged = modern_results.merge(
    modern_races[['raceId', 'year', 'round', 'circuitId', 'name']], on='raceId'
).merge(driver_info, on='driverId')

driver_stats = merged.groupby('full_name').agg(
    races      = ('raceId', 'count'),
    wins       = ('positionOrder', lambda x: (x == 1).sum()),
    podiums    = ('positionOrder', lambda x: (x <= 3).sum()),
    avg_finish = ('positionOrder', 'mean'),
    total_pts  = ('points', 'sum'),
    avg_pts    = ('points', 'mean')
).reset_index().sort_values('total_pts', ascending=False)

print(f"\nTop 10 Drivers (2020-2024 by total points):")
print(driver_stats.head(10).to_string(index=False))

# ─────────────────────────────────────────────────────────────
# 2. PIT STOP STATS SUMMARY
# ─────────────────────────────────────────────────────────────

pit_summary = modern_pitstops.groupby('raceId').agg(
    avg_stops     = ('stop', 'max'),
    avg_stop_time = ('milliseconds', 'mean')
).reset_index()

pit_summary['avg_stop_time_sec'] = pit_summary['avg_stop_time'] / 1000

overall_pit = {
    'avg_stops_per_race'    : pit_summary['avg_stops'].mean(),
    'avg_pitstop_time_sec'  : pit_summary['avg_stop_time_sec'].mean(),
    'fastest_pitstop_sec'   : modern_pitstops['milliseconds'].min() / 1000,
    'slowest_pitstop_sec'   : modern_pitstops['milliseconds'].max() / 1000
}

print(f"\nPit Stop Stats (2020-2024):")
for k, v in overall_pit.items():
    print(f"  {k}: {v:.2f}")

# ─────────────────────────────────────────────────────────────
# 3. CONSTRUCTOR STATS SUMMARY
# ─────────────────────────────────────────────────────────────

const_info   = constructors[['constructorId', 'name']].copy()
const_merged = modern_results.merge(
    modern_races[['raceId', 'year']], on='raceId'
).merge(const_info, on='constructorId')

const_stats = const_merged.groupby('name').agg(
    wins    = ('positionOrder', lambda x: (x == 1).sum()),
    podiums = ('positionOrder', lambda x: (x <= 3).sum()),
    total_pts = ('points', 'sum'),
    avg_pts   = ('points', 'mean')
).reset_index().sort_values('total_pts', ascending=False)

print(f"\nTop 5 Constructors (2020-2024):")
print(const_stats.head(5).to_string(index=False))

# ─────────────────────────────────────────────────────────────
# 4. BUILD AI CONTEXT STRING
# ─────────────────────────────────────────────────────────────

top_drivers    = driver_stats.head(10)
top_const      = const_stats.head(5)

context = f"""
You are an expert F1 Race Engineer AI with deep knowledge of 
Formula 1 strategy, tyre management, pit stop timing, and race craft.
You have access to real F1 data from the 2020-2024 seasons.

REAL F1 DATA SUMMARY (2020-2024):

TOP 10 DRIVERS BY POINTS:
{top_drivers[['full_name','wins','podiums','avg_finish','total_pts']].to_string(index=False)}

TOP 5 CONSTRUCTORS BY POINTS:
{top_const[['name','wins','podiums','total_pts']].to_string(index=False)}

PIT STOP STATISTICS:
- Average stops per race    : {overall_pit['avg_stops_per_race']:.2f}
- Average pit stop time     : {overall_pit['avg_pitstop_time_sec']:.2f} seconds
- Fastest pit stop recorded : {overall_pit['fastest_pitstop_sec']:.2f} seconds
- Slowest pit stop recorded : {overall_pit['slowest_pitstop_sec']:.2f} seconds

TOTAL RACES ANALYZED: {modern_races.shape[0]} (2020-2024)
TOTAL RACE ENTRIES  : {modern_results.shape[0]}

INSTRUCTIONS:
- Always answer as a professional F1 Race Engineer
- Use the real data above to support your answers
- Give clear, specific strategy advice when asked
- Reference real drivers, teams, and statistics when relevant
- Keep answers focused and professional
- If asked about tyre strategy, pit windows, or race tactics give detailed answers
"""

print("\n" + "="*60)
print("AI Context built successfully!")
print(f"Context length: {len(context)} characters")
print("="*60)
print("\nSample of context sent to AI:")
print(context[:500] + "...")

Loading F1 data for AI Race Engineer...
Races loaded       : 107
Results loaded     : 2139
Pit stops loaded   : 3934
Qualifying loaded  : 2138

Top 10 Drivers (2020-2024 by total points):
      full_name  races  wins  podiums  avg_finish  total_pts   avg_pts
 Max Verstappen    107    55       81    3.775701     1964.5 18.359813
 Lewis Hamilton    106    21       51    5.103774     1389.5 13.108491
Charles Leclerc    107     6       33    7.000000     1060.0  9.906542
   Sergio Pérez    105     6       31    7.200000     1004.0  9.561905
   Carlos Sainz    106     4       26    7.207547      936.5  8.834906
   Lando Norris    107     4       26    7.168224      901.0  8.420561
 George Russell    107     3       15    9.495327      664.0  6.205607
Valtteri Bottas    107     3       22   11.252336      499.0  4.663551
Fernando Alonso     90     0        9    9.411111      430.0  4.777778
  Oscar Piastri     46     2       10    7.891304      347.0  7.543478

Pit Stop Stats (2020-2024):
  